# Agent Orchestrator and Tools Usage Guide

This notebook provides a structured guide on how to initialize and use the individual tools and the Agent Orchestrator inside the Kapruka Gift Concierge project.

In [1]:
import sys
import os

sys.path.append(os.path.abspath('..'))


## 1. Using the Router
The Router agent decides the execution path based on the user's query.

In [4]:
from src.agents.router import Router
from src.memory.semantic_memory_manager import SemanticMemoryManager
from src.memory.short_term_memory_manager import ShortTermMemoryManager
from src.infrastructure.llm.llm_provider import get_chat_llm

user_id = '1'
user_query = 'my mother is allergy to nuts. suggest some gifts for her. Deliver to panadura.'
st_memory = ShortTermMemoryManager()

router = Router(llm=get_chat_llm(), semantic_memory=SemanticMemoryManager())
res = router.route(user_query, st_memory, user_id)
print(res)

update_profile=False search_catalog=True check_logistics=True direct_chat=False target_location='Panadura' vector_query='suggest some gifts for my mother who is allergic to nuts' keyword_query='cakes'


## 2. Using the Tools
The system has several tools that handle specific capabilities.

### 2.1 Preference Update Tool
Updates the semantic memory profile of a user based on their message.

In [11]:
from src.agents.tools.preference_update_tool import PreferenceUpdateTool
from src.memory.semantic_memory_manager import SemanticMemoryManager

sem_memory = SemanticMemoryManager()
pref_tool = PreferenceUpdateTool(llm=get_chat_llm(), semantic_memory=sem_memory)

pref_tool.update_semantic_memory(user_id='2', user_message='my father likes golf and chocolate')

print(sem_memory.get_profile('2'))

2026-04-19 01:29:32.164 | INFO     | src.memory.semantic_memory_manager:save_profile:32 - Semantic Memory updated for 2.


{'wife': {'preferences': ['mango'], 'allergies': ['nuts']}, 'father': {'preferences': ['golf', 'chocolate'], 'allergies': []}}


### 2.2 Logistics Tool
Checks delivery feasibility for specific products to certain locations.

In [7]:
from src.agents.tools.logistics_tool import LogisticsTool
from src.infrastructure.llm.llm_provider import get_chat_llm

logistics_tool = LogisticsTool(llm=get_chat_llm())
logistics_res = logistics_tool.check_delivery_feasibility("Matara", "ice cream")
print(logistics_res)

deliverable=False reason='Ice cream is a highly sensitive item that cannot be shipped to distant districts due to the risk of melting.'


### 2.3 Catalog Search Tool
Performs hybrid search (vector + keyword) over the product catalog.

In [9]:
from src.agents.tools.catalog_search_tool import CatalogSearchTool

catalog_tool = CatalogSearchTool()

catalog_res = catalog_tool.search(
    vector_query="suggest some gifts for my mother who is allergic to nuts",
    keyword_query="cakes"
)
print(catalog_res)

[{'title': 'Moms Tasteful Treats Gift Set', 'description': '\n\n\nThe Moms Tasteful Treats Gift Set is the perfect surprise for any mom. Available at Kapruka in Sri Lanka, this set is packed with delightful goodies to show just how much you care.\n\nJar of Nuts (Large): A healthy and tasty snack option.\nPack of 3 Ferrero Rocher: Creamy chocolates with a crunchy bite.\nJar of Olives: Perfect for those who love savory treats.\nToblerone Chocolate Bar: Swiss chocolate with honey and almond nougat.\nPack of Organic Tea: Relaxing and refreshing herbal tea.\n1 Kit Kat 4 Finger: A sweet break treat (Motherâ€™s Day Cookies not included).\nOrganic Jam Jar: Tasty spread for bread and pastries.\nPack of Ground Coffee: Fresh Ceylon coffee for a perfect morning start.\nComplimentary Card: A heartfelt way to express your love.\n\nImportant Notice: Product brands or ranges will depend on availability. If a specific item is unavailable, we may substitute it with a comparable flavor, fragrance, or bra

### 2.4 Direct Chat Tool
Handles straightforward conversational replies when tool execution is unnecessary.

In [10]:
from src.agents.tools.direct_chat_tool import DirectChatTool
from src.memory.short_term_memory_manager import ShortTermMemoryManager

st_memory = ShortTermMemoryManager()

chat_tool = DirectChatTool(llm=get_chat_llm())
chat_res = chat_tool.chat("Hi", st_memory)
print(chat_res)

Hello! How can I assist you today? If you're looking for a special gift, cake, or flower arrangement, I'm here to help you find the perfect option on Kapruka!


## 3. Running the Agent Orchestrator
The `AgentOrchestrator` integrates the Router, Tools, and Reflection Agent to automatically manage user requests.

In [13]:
from IPython.core.display import Markdown
from src.agents.orchestrator import build_orchestrator
from src.memory.short_term_memory_manager import ShortTermMemoryManager


st = ShortTermMemoryManager()
agent = build_orchestrator()

### 3.1 Initial Request
The Orchestrator routes the query, checks tools, and refines the response via the Reflection Agent.

In [17]:
query = "my mother likes flowers. suggest some gifts for her. Deliver to panadura."
res = agent.chat(user_id='2', user_query=query, st_memory=st)
Markdown(res)

2026-04-19 01:32:30.046 | INFO     | src.agents.orchestrator:chat:45 - Processing query for user 2: my mother likes flowers. suggest some gifts for her. Deliver to panadura.
2026-04-19 01:32:30.047 | INFO     | src.agents.orchestrator:_status:41 - Analyzing request and making routing decisions...
2026-04-19 01:32:31.611 | INFO     | src.agents.orchestrator:_status:41 - Routing decision: update_profile=True search_catalog=True check_logistics=True direct_chat=False target_location='Panadura' vector_query='suggest some gifts for my mother who likes flowers and deliver to Panadura' keyword_query='flowers, gifts'
2026-04-19 01:32:31.611 | INFO     | src.agents.orchestrator:_status:41 - Analyzing and updating user profile preferences...
2026-04-19 01:32:31.613 | INFO     | src.agents.orchestrator:_status:41 - Searching Kapruka catalog for best matches...
2026-04-19 01:32:31.616 | INFO     | src.agents.orchestrator:_status:41 - Checking logistics and delivery feasibility...
2026-04-19 01:32:

Here are some lovely gift options for your mother, who adores flowers and has a nut allergy. All of these can be delivered to Panadura:

1. **Sweet Embrace Combo For Mom With Roses Bouquet**  
   Celebrate your mom's love with this delightful combo that includes a beautiful Pink Serenity Bouquet featuring roses, carnations, and more. It also comes with a charming greeting card to express your love.  
   **Price:** RS. 15,160  
   **[View Sweet Embrace Combo](https://www.kapruka.com/buyonline/sweet-embrace-combo-for-mom-wi/kid/combogifl41)**

2. **Little Joys Gift Set With Flowers**  
   Brighten her day with this gift set that combines fresh flowers with a cute teddy. It's elegantly packaged and perfect for any occasion.  
   **Price:** RS. 11,700  
   **[View Little Joys Gift Set](https://www.kapruka.com/buyonline/little-joys-gift-set-with-flow/kid/combogifl82)**

3. **Blush Of Love Bouquet With Two Red And Five Pink Roses**  
   This beautiful bouquet features a mix of red and pink roses, along with additional flowers, wrapped delicately. It's a heartfelt gift that expresses love and admiration.  
   **Price:** RS. 5,850  
   **[View Blush Of Love Bouquet](https://www.kapruka.com/buyonline/blush-of-love-bouquet-with-two/kid/flowers00t1900)**

4. **Lovely Mother Zebra Cactus Pot With Tag**  
   If she enjoys plants, this charming zebra cactus pot is a unique gift that requires minimal care and adds a touch of nature to her space. It comes with a tag expressing gratitude to Mom.  
   **Price:** RS. 3,350  
   **[View Zebra Cactus Pot](https://www.kapruka.com/buyonline/lovely-mother-zebra-cactus-pot/kid/ef_pc_flow0v2383p00006)**

5. **Thanks Mom Greeting Card**  
   Pair any of the above gifts with this beautiful greeting card that expresses your gratitude and love for her.  
   **Price:** RS. 550  
   **[View Thanks Mom Card](https://www.kapruka.com/buyonline/thanks-mom-greeting-card/kid/greeting00z2339)**

All these options are nut-free and perfect for your mother. Let me know if you would like to choose one or need more assistance!

### 3.2 Conversational Follow-up
Maintains conversational context using short-term memory.

In [18]:
follow_up_query = "thank you i choose: Sweet Embrace Combo For Mom With Roses Bouquet"
res = agent.chat('2', follow_up_query, st)
Markdown(res)

2026-04-19 01:33:26.629 | INFO     | src.agents.orchestrator:chat:45 - Processing query for user 2: thank you i choose: Sweet Embrace Combo For Mom With Roses Bouquet
2026-04-19 01:33:26.630 | INFO     | src.agents.orchestrator:_status:41 - Analyzing request and making routing decisions...
2026-04-19 01:33:28.296 | INFO     | src.agents.orchestrator:_status:41 - Routing decision: update_profile=False search_catalog=True check_logistics=True direct_chat=True target_location='Panadura' vector_query='suggest gifts for my mother who likes flowers and has a nut allergy, deliver to Panadura' keyword_query='flowers, bouquet, gift set'
2026-04-19 01:33:28.296 | INFO     | src.agents.orchestrator:_status:41 - Preparing direct response...


You're very welcome! The **Sweet Embrace Combo For Mom With Roses Bouquet** is a beautiful choice that I'm sure your mother will love. If you need any further assistance with the order or have any special requests for the delivery, please let me know!